# ASHA Sathi - Maternal ANC Data Exploration

EDA on the synthetic maternal dataset generated by `src.data.loaders`. Run `python -m src.training.pipeline --stage prepare_data` first to produce `data/processed/maternal.parquet`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 40)

In [ ]:
from src.data.loaders import generate_synthetic_maternal

df = generate_synthetic_maternal(n=5000, seed=42)
df.head()

In [ ]:
print(df.shape)
df.info()

In [ ]:
missing = df.isna().sum()
print("Missing values per column:")
print(missing[missing > 0])

In [ ]:
ax = sns.countplot(data=df, x="risk_level", order=["low", "medium", "high"])
ax.set_title("Distribution of maternal risk levels")
for container in ax.containers:
    ax.bar_label(container)
plt.show()

In [ ]:
num_cols = [
    "age", "gravida", "parity", "gestational_weeks", "bp_systolic",
    "bp_diastolic", "hemoglobin", "blood_sugar", "bmi", "anc_visits_attended",
]
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="teal")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[num_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0)
plt.title("Correlation matrix of continuous features")
plt.show()

In [ ]:
key_vitals = ["bp_systolic", "bp_diastolic", "hemoglobin", "blood_sugar", "bmi"]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, key_vitals):
    sns.boxplot(data=df, x="risk_level", y=col, order=["low", "medium", "high"], ax=ax)
    ax.set_title(f"{col} by risk level")
plt.tight_layout()
plt.show()

In [ ]:
binary_cols = [
    "previous_csection", "previous_pph", "previous_preterm",
    "previous_stillbirth", "htn", "diabetes", "heart_disease",
    "kidney_disease", "pmsma_attended",
]
rates = df.groupby("risk_level")[binary_cols].mean().T
rates.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
rates.plot(kind="bar", ax=ax)
ax.set_ylabel("Proportion with risk factor")
ax.set_title("Prevalence of maternal risk factors by risk level")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Risk level")
plt.tight_layout()
plt.show()

## Summary

* Classes are imbalanced (most pregnancies are low-risk), so the training loop uses `compute_class_weight`.
* Vital signs (`bp_systolic`, `hemoglobin`) separate risk levels cleanly, matching the labelling rule.
* Feature engineering imputes the ~3% missing values and standardises continuous columns before XGBoost.